# 11 · Copies, views, strides, and memory

**Level:** Advanced · **Practice time:** 30–50 minutes

[← Previous](10_linear_algebra.ipynb) · [Course guide](../README.md) · [Next →](12_vectorization_and_performance.ipynb)

**You will learn:** Predict shared memory, inspect strides, and use a memory-mapped array.

**Prerequisites:** Notebooks 00–10.

## How to learn with this notebook

1. Run cells from top to bottom. Predict results before executing each example.
2. Do the **Try it** experiment by editing the example or adding a cell below it.
3. Replace `None` in each exercise with your answer, then rerun its check cell.
4. Consult the solutions at the end only after trying. An untouched exercise prints **Pending**, so Run All succeeds without claiming you solved it.
5. Complete the independent challenge, then restart the kernel and run all cells again.

Each notebook imports its own dependencies and creates its own data. You do not need variables from another notebook. The supplied outputs show a verified reference run; rerun cells to explore your changes.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## Step 1 · Basic slicing shares data

A view refers to an existing data buffer. Basic slicing normally produces a view; `.copy()` makes independent storage. Changing a view changes the original data. `shares_memory` checks whether two arrays overlap in memory.

In [2]:
original = np.arange(6)
view = original[1:4]
independent = original[1:4].copy()
view[0] = 99
print("Original:", original)
print("Independent copy:", independent)
assert np.shares_memory(original, view)
assert not np.shares_memory(original, independent)
assert original[1] == 99

Original: [ 0 99  2  3  4  5]
Independent copy: [1 2 3]


**Try it:** Modify independent and confirm original stays unchanged.

## Step 2 · Advanced indexing and strides

Integer-array and Boolean indexing return copies when reading. Assignment like `a[mask] = value` still writes to a. Avoid chained indexing when you intend to modify the original. Strides are byte jumps between adjacent entries along each axis. A transpose can reorder strides without moving the values.

In [3]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)
selected = a[[0, 2]]
selected[0, 0] = -100
assert a[0, 0] == 0
print("Original strides:", a.strides)
print("Transpose strides:", a.T.strides)
print("C contiguous:", a.flags.c_contiguous, a.T.flags.c_contiguous)
flat_copy = a.flatten()
flat_maybe_view = a.ravel()
assert not np.shares_memory(a, flat_copy)
assert np.shares_memory(a, flat_maybe_view)
# reshape/ravel may need a copy for other layouts: inspect rather than assume.

Original strides: (32, 8)
Transpose strides: (8, 32)
C contiguous: True False


**Try it:** Compare shares_memory(a, a.T.reshape(-1)) with shares_memory(a, a.reshape(-1)).

## Step 3 · Read a file through memory mapping

Memory mapping exposes file-backed array storage without first reading the whole file into a normal array. Operations can still allocate full-size outputs, so process slices when data is large. Here we use a small temporary file and keep the read-only mapping within its lifetime.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as folder:
    path = Path(folder) / "values.npy"
    np.save(path, np.arange(1000, dtype=np.float64))
    mapped = np.load(path, mmap_mode="r", allow_pickle=False)
    first_chunk_mean = mapped[:100].mean()
    print("Mapped type:", type(mapped).__name__)
    print("First chunk mean:", first_chunk_mean)
    np.testing.assert_allclose(first_chunk_mean, 49.5)
    del mapped  # release the file mapping before temporary directory cleanup on Windows

Mapped type: memmap
First chunk mean: 49.5


**Try it:** Compute the mean using ten chunks and compare with the full-array result.

## Your practice exercises

Use the named answer variables. The checks verify your result; they do not require the same approach as the reference solution. If an assertion fails, read the expected value and inspect your shape and dtype.

### Exercise 1

Take an independent copy of the first two rows of source.

In [5]:
source = np.arange(12).reshape(4, 3)

safe_rows = None  # TODO: replace None with your answer

In [6]:
if safe_rows is None:
    print("Pending: complete Exercise 1 first.")
else:
    np.testing.assert_array_equal(safe_rows, [[0, 1, 2], [3, 4, 5]])
    assert not np.shares_memory(source, safe_rows)
    print("Exercise 1 passed!")

Pending: complete Exercise 1 first.


### Exercise 2

Take every second element as a view, without copying.

In [7]:
source_vector = np.arange(10)

every_second = None  # TODO: replace None with your answer

In [8]:
if every_second is None:
    print("Pending: complete Exercise 2 first.")
else:
    np.testing.assert_array_equal(every_second, [0, 2, 4, 6, 8])
    assert np.shares_memory(source_vector, every_second)
    print("Exercise 2 passed!")

Pending: complete Exercise 2 first.


## Independent challenge

Compare the shape, strides, contiguity flags, and sharing behavior of a matrix, its transpose, a stepped slice, and an explicitly contiguous copy.

Add code and explanation cells below. Decide on expected results before checking your implementation.

In [9]:
# Your independent challenge work goes here.

## Before you move on

- Explain the main idea in your own words without reading the examples.
- Make both exercise checks print **passed**.
- Try at least one changed input or edge case and explain the result.
- Finish the independent challenge or write down a specific question to revisit.

Expertise comes from repeated practice and debugging unfamiliar problems. Completing these notebooks gives you a broad practical foundation, not a substitute for that experience.

---

## Reference solutions — try first

These cells use separate reference variables, so they do not fill in your exercise answers. They also check the reference result. Compare your reasoning as well as the output.

### Solution 1

Take an independent copy of the first two rows of source.

**Why it works:** The slice chooses the rows, and copy separates their storage from the source.

In [10]:
reference_safe_rows = source[:2].copy()
np.testing.assert_array_equal(reference_safe_rows, [[0, 1, 2], [3, 4, 5]])
assert not np.shares_memory(source, reference_safe_rows)
print(reference_safe_rows)

[[0 1 2]
 [3 4 5]]


### Solution 2

Take every second element as a view, without copying.

**Why it works:** A step of 2 skips every other entry. Basic slicing shares the underlying buffer.

In [11]:
reference_every_second = source_vector[::2]
np.testing.assert_array_equal(reference_every_second, [0, 2, 4, 6, 8])
assert np.shares_memory(source_vector, reference_every_second)
print(reference_every_second)

[0 2 4 6 8]


## Continue learning

- [Official NumPy documentation: basics.copies ](https://numpy.org/doc/stable/user/basics.copies.html)
- [Official NumPy documentation: numpy.memmap ](https://numpy.org/doc/stable/reference/generated/numpy.memmap.html)

[Next →](12_vectorization_and_performance.ipynb)